Builds the HuggingFace dataset in `../hf_dataset` (outside the repo, rebuilt from scratch).
Pins = all pins in `main_run/answers.parquet`; sources and boxes restricted to those. Images are APFS clones (`cp -c`).
Upload: `cd ../hf_dataset && hf upload phillip-sievers/GeoPrecisionBench . --repo-type=dataset`

In [1]:
import shutil
import subprocess

import duckdb
import geopandas as gpd
import pandas as pd

from thesis.config import REPO_ROOT

src = REPO_ROOT / "data" / "dataset_v2"
dst = REPO_ROOT.parent / "hf_dataset"
README = """---
license: odbl
pretty_name: GeoPrecisionBench
---

# GeoPrecisionBench

TODO: description, citation.

| file | content |
|---|---|
| `pins.parquet` | one row per pin; `image` = path of the render the models saw |
| `sources.parquet` | one row per source building |
| `boxes.parquet` | sampling boxes |
| `answers.parquet` | main-run model answers (verification, distance) |
| `consistency_answers.parquet` | repeats 1-4 on the human-comparison pins |
| `gold/` | building check, human on-roof answers and review, failure-mode codes |
| `configs/` | prompts, bins, scoring settings (incl. excluded sources) |

Footprints: Overture Maps 2026-08-19.0. Imagery: USDA NAIP.
"""

In [2]:
answers = pd.read_parquet(src / "main_run" / "answers.parquet")
with duckdb.connect(str(src / "manifest.duckdb"), read_only=True) as connection:
    renders = connection.execute("SELECT pin_id, render_id, image_path, image_sha256 FROM renders").fetch_df()
pins = pd.read_parquet(src / "audit" / "selected_pins.parquet")
pins = pins.loc[pins["pin_id"].isin(answers["pin_id"])].merge(renders, on="pin_id", validate="1:1")
# HF allows at most 10,000 files per directory: shard by the first two hex chars of render_id
pins["image"] = "images/" + pins["render_id"].str[:2] + "/" + pins["render_id"] + ".png"
assert len(pins) == answers["pin_id"].nunique()
sources = pd.read_parquet(src / "selection" / "sources.parquet").loc[
    lambda f: f["source_id"].isin(answers["source_id"])
]
boxes = gpd.read_parquet(src / "inventory" / "boxes.parquet").loc[lambda f: f["box_id"].isin(sources["box_id"])]
len(pins), len(sources), len(boxes), len(answers)

(14398, 1832, 119, 86388)

In [3]:
if dst.exists():
    shutil.rmtree(dst)
(dst / "images").mkdir(parents=True)
(dst / "gold").mkdir()
(dst / "configs").mkdir()

pins.drop(columns="image_path").to_parquet(dst / "pins.parquet")
sources.to_parquet(dst / "sources.parquet")
boxes.to_parquet(dst / "boxes.parquet")
answers.to_parquet(dst / "answers.parquet")
shutil.copy(src / "consistency_answers.parquet", dst)
for name in ["is_building", "human_on_roof", "human_error_review", "failure_modes"]:
    shutil.copy(src / "gold" / f"{name}.jsonl", dst / "gold")
for name in ["prompts", "scoring", "bins"]:
    shutil.copy(REPO_ROOT / "configs" / f"{name}.yaml", dst / "configs")
(dst / "README.md").write_text(README)

for image_path, image in zip(pins["image_path"], pins["image"]):
    (dst / image).parent.mkdir(exist_ok=True)
    subprocess.run(["cp", "-c", src / image_path, dst / image], check=True)
len(list((dst / "images").rglob("*.png"))), max(len(list(d.iterdir())) for d in (dst / "images").iterdir())

(14398, 82)